# 00. Start with the research question and the real cohort

We want to learn a reference for normal movement, predict what follows a short observed past, and investigate whether prediction errors and learned representations help identify gait deviations. This lesson separates that research question from a clinical diagnosis claim. By the end, you will have rebuilt the annotation inventory and checked a source-disjoint partition.

## Run locally

Run this notebook independently from the experiment directory. No previous notebook execution is required. The environment installs this local `worldmodels` package and the repository's editable Ambient package together.

```bash
cd experiments/aaai-worldmodels
uv sync
uv run python -m ipykernel install --user --name medwm-gait --display-name "Med-WM gait (Python 3.12)"
uv run jupyter lab
```

Select **Med-WM gait (Python 3.12)**. The project requires Python 3.12. Place local settings in this project's ignored `.env`. The requested `load_env(find_env())` spelling uses aliases for `python-dotenv`; it resolves this experiment's `.env` explicitly and never displays credentials. These lessons need no paid model API.

Set `WM_DATA_MODE=real` to use only actual prepared GAVD skeletons and real MediaPipe/PyTorch/classifier calls. Real mode requires the immutable person-gated clean cohort and fails if those inputs are absent; it never substitutes synthetic data. It embeds a short local walking excerpt and persists numerical outputs and trained lesson checkpoints under a separately named real execution. Run all real lessons with `uv run python scripts/execute_notebooks.py --data-mode real`.

The default `synthetic` mode is an optional fast teaching fixture. Its motions are not medical simulations and cannot establish clinical accuracy. Real lesson training is also a bounded tutorial exercise, distinct from the primary controlled study. Keep all three evidence types separate.

In [ ]:
from pathlib import Path
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
from IPython.display import display, Markdown, SVG
from worldmodels.settings import PROJECT, REPOSITORY, load_env, find_env
load_env(find_env())
torch.set_num_threads(2)
from worldmodels.training import seed_everything
from worldmodels.notebook_support import (data_mode, lesson_data, real_inputs, real_example,
    bounded_preview, lesson_directory, persist_lesson, prepared_path, comparison_path)
REAL = data_mode() == "real"
RELEASE_TEST_RESULTS = os.getenv("WM_RELEASE_TEST_RESULTS", "0") == "1"
EVIDENCE_LABEL = "Real GAVD" if REAL else "Synthetic teaching"
from matplotlib_inline.backend_inline import set_matplotlib_formats
set_matplotlib_formats("svg")
seed_everything(27)
print("Data mode:", data_mode(), "| Prepared source:", prepared_path() if REAL else "synthetic fixture")
print("Project:", PROJECT.name, "| Python kernel ready")

### Walking-video context

Real mode embeds only a short original walking excerpt from an accepted normal training sequence. The preview is saved under `data/videos-gavd/previews/`, with its source hash and frame range. It provides context for this lesson without duplicating the full original. Synthetic mode enables the same real viewing aid only when `WM_EMBED_VIDEO=1`.

In [ ]:
if REAL or os.getenv("WM_EMBED_VIDEO", "0") == "1":
    from IPython.display import Video
    preview_path = bounded_preview()
    display(Video(str(preview_path), embed=True, html_attributes='controls preload="metadata"'))
    print("Bounded source excerpt:", preview_path.relative_to(REPOSITORY))
else:
    print("Synthetic mode: set WM_EMBED_VIDEO=1 for a real-video context excerpt.")

## Step 1. Decide what the experiment can answer

A skeleton is a time series of estimated landmarks. A predictive representation can learn recurring relationships between limbs without being given diagnostic labels. A classifier can then use the learned representation to predict the supplied dataset label. These are different tasks: useful classification does not prove forecasting, and accurate short-horizon forecasting does not prove clinical understanding.

The primary supervised comparison is normal versus the four selected non-normal labels, evaluated with source-level balanced accuracy. Five-class macro-F1 is secondary. The anomaly branch keeps a normal-only reference and uses future prediction errors. Folder labels remain annotations, not verified diagnoses.

In [ ]:
display(SVG(filename=str(PROJECT / "images/cohort.svg")))

## Step 2. Rebuild from current CSVs

We read current per-sequence CSVs and exclude `data/gavd/source/`, which contains the same upstream rows in their original parts. Counting both would double-count observations. Each row below represents one annotated sequence, not one patient. The inventory checks schema, sequence identity, source identity, label consistency, and duplicate frame indexes.

In [ ]:
from worldmodels.data import build_inventory, assign_splits, LABELS
inventory, audit = build_inventory(REPOSITORY)
display(pd.DataFrame(audit["by_label"]).T)
print({key: audit[key] for key in ("all_sequences", "all_rows", "all_sources", "selected_sequences", "selected_sources")})
assert tuple(LABELS) == ("normal", "parkinsons", "stroke", "cerebral palsy", "myopathic")

## Step 3. Count the independent units

The selected cohort contains 666 sequences from 103 YouTube sources before media attrition. There are 140,641 annotation rows. These are different denominators. Cutting a source into many clips increases training examples without creating independent patients. True person identity and edited repost relationships are not established by a YouTube identifier.

Stroke was repeated in the brief and is included once. Myopathic gait motivates a neuromuscular question; it should not automatically be described as a central neurological condition. Restricting the first study to these five labels makes its scope clear, but excludes many other forms of abnormal gait.

In [ ]:
selected_counts = inventory.groupby("label").agg(sequences=("sequence_id", "size"), sources=("source_id", "nunique"), rows=("n_frames", "sum"))
display(selected_counts)
display(inventory.loc[~inventory.eligible, ["label", "source_id", "sequence_id", "exclusion_reason"]])
print("Annotation identity:", audit["annotation_identity"])

## Step 4. Split sources before training

All sequences from one source inherit one split. This prevents a model from learning from one part of a video and being tested on another part of the same recording. The split is established before pose quality filtering. We do not replace an excluded test source with a more convenient one after seeing performance.

Training fits the representation and classifiers. Validation chooses declared development settings and thresholds. Test sources are reserved for final evaluation. Unlabeled test motion is still test data and must not enter self-supervised training.

In [ ]:
display(SVG(filename=str(PROJECT / "images/source_split.svg")))

In [ ]:
partition = assign_splits(inventory, seed=27)
assert partition.groupby("source_id").split.nunique().max() == 1
display(partition.loc[partition.eligible].groupby(["split", "label"]).agg(sources=("source_id", "nunique"), sequences=("sequence_id", "size")))
if REAL:
    _, _, clean_metadata = real_inputs()
    print("Active person-gated clean cohort, after pose and source exclusions:")
    display(clean_metadata.groupby(["split", "label"]).agg(sources=("source_id", "nunique"), windows=("sequence_id", "size")))

## Step 5. Interpret the limitations before a score exists

The fixed split has only one media-eligible cerebral-palsy test source. Its many sequences cannot estimate between-person variation. The broader GAVD inventory has 348 sources, but this five-label study has far fewer, and missing media reduces it further. Source grouping is necessary but cannot by itself establish generalization to new patients, clinics, diagnoses, or camera systems.

A credible workshop claim is an auditable feasibility study of predictive gait representations under source shifts. A stronger clinical claim needs independently verified labels and external patient/site evaluation. The supplied neurological framing can motivate hypotheses without turning uncertain annotations into clinical ground truth.

In [ ]:
display(Markdown((PROJECT / "docs/workshop.md").read_text()))

## Step 6. Freeze evidence without silently overwriting it

The command below records the exact current CSV identity and source partition. Changed inputs require a new output directory. This behavior protects earlier evidence from being relabeled after a method or cohort changes.

In [ ]:
from worldmodels.data import save_inventory
# Reusing the same unchanged inventory is safe; changed evidence refuses overwrite.
report = save_inventory(partition, audit, PROJECT / "artifacts/runs/notebook-inventory")
print("Frozen partition:", report["partition_sha256"])

In [ ]:
lesson_evidence = persist_lesson("00_inventory_and_research_question", globals())
print("Evidence mode:", lesson_evidence.get("mode"))

## What to retain

Record the data identity, settings, executed commands, and limitations with any result. A successful cell execution is implementation evidence, not proof that the method generalizes. See the [data protocol](docs/data_protocol.md), [method review](docs/literature.md), and [README](README.md) before changing the study.